# 02 · Run the general-LLM baselines

Same prompts, same sampling budget, same adapter, same metrics as the specialised systems.

The first three models are the **base models** of CADmium, CADFusion and cadrille. Running
them turns the benchmark into three controlled ablations — same weights, same prompts, the
only difference being the CAD fine-tune. That comparison is the strongest claim this
benchmark can make and it costs nothing extra.

In [ ]:
# Mount Drive so predictions survive a Colab timeout, then get the harness.
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/t2c_bench'
os.makedirs(WORK, exist_ok=True)
os.environ['T2C_WORK'] = WORK

!git clone -q https://github.com/prashantkul366/T2C_Benchamrk /content/t2cbench_repo || (cd /content/t2cbench_repo && git pull -q)
%cd /content/t2cbench_repo
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
print('work dir:', WORK)

In [ ]:
#@title Configuration
MODEL = 'qwen25-coder-7b'  #@param ['qwen25-coder-7b','llama3-8b-instruct','qwen2-vl-2b','mistral-7b-instruct','deepseek-coder-6.7b','qwen25-coder-32b']
SPLIT = 'A'                #@param ['A','B']
SHOT  = 'general_one_shot' #@param ['general_one_shot','general_zero_shot']
MODE  = 'pass_at_1'        #@param ['pass_at_1','best_of_k']

import yaml, os
cfg = yaml.safe_load(open('configs/models.yaml'))[MODEL]
tag = MODEL + ('_0shot' if SHOT.endswith('zero_shot') else '')
SPLIT_FILE = f"{os.environ['T2C_WORK']}/data/split_{SPLIT.lower()}.jsonl"
OUT = f"{os.environ['T2C_WORK']}/results/raw/{tag}_split{SPLIT}_{MODE}.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
print(yaml.dump(cfg, sort_keys=False)); print('output:', OUT)

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
import torch; print(torch.cuda.get_device_name(0))
# llama3-8b-instruct is gated -- accept Meta's licence, then log in.
# from huggingface_hub import login; login()

In [ ]:
q4 = '--load-4bit' if cfg.get('load_4bit') else ''
!python -m t2cbench.runners.run_hf \
    --model {cfg['weights']} --name {tag} --template {SHOT} {q4} \
    --split {SPLIT_FILE} --out {OUT} --mode {MODE} --batch-size 8

### Look at what it actually produced

Worth eyeballing: general LLMs fail in ways the fine-tuned systems do not — prose outside the
code fence, a `Sketch` that is never extruded, imports of libraries that are not installed.
The 8-way validity taxonomy in notebook 03 separates those from real geometric errors.

In [ ]:
import json
rows = [json.loads(l) for l in open(OUT)]
print(f'{len(rows)} generations')
for r in rows[:2]:
    print('='*70); print(r['output'][:700])